# Gold – Member A: demand & resolution questions

- **Q1** Where service demand is concentrated
- **Q2** Where resolution performance is slow or inconsistent

**Caching:** Level 1 = `base`. No Level-2 cache needed: each table is one `groupBy` on the cached base.

## 1. Shared setup
Same code as **section 1 of `gold.py`** – don't change it here without changing it there too.

In [3]:
import findspark
findspark.init()

from pyspark import StorageLevel
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [5]:
DEFAULT_OUTPUT_DIR = "/home/ubuntu/spark-project-data/dev"
DEFAULT_SILVER_PATH = f"{DEFAULT_OUTPUT_DIR}/silver/nyc311_enriched"
DEFAULT_GOLD_OUTPUT = f"{DEFAULT_OUTPUT_DIR}/gold"
GOLD_BASE_COLUMNS = [
    # request
    "request_id", "created_at", "closed_at", "request_year", "request_month",
    "status", "resolution_hours", "lifecycle_quality_status",
    # what / who
    "complaint_type", "agency",
    # where (Member A)
    "borough", "community_board", "incident_zip",
    # property (Member B)
    "bbl", "pluto_match_status", "pluto_land_use", "pluto_building_class",
    "pluto_residential_units", "pluto_yearbuilt",
]


def create_spark_session(app_name="DAT535-NYC311-PLUTO-Gold", cores_max="12"):
    """Spark session for the cluster."""
    spark = (
        SparkSession.builder
        .appName(app_name)
        .master("spark://192.168.18.168:7077")
        .config("spark.driver.host", "192.168.18.168")
        .config("spark.cores.max", cores_max)
        .config("spark.executor.cores", "4")
        .config("spark.executor.memory", "6g")
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.sql.shuffle.partitions", "32")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    spark.conf.set("spark.sql.parquet.int96RebaseModeInWrite", "CORRECTED")
    spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")
    return spark


def load_gold_base(spark, silver_path=DEFAULT_SILVER_PATH, sample_pct=None):
    """LEVEL-1 CACHE: read Silver once, select Gold columns, persist.

    sample_pct: e.g. 1 = deterministic 1% of requests (same rows every run),
                useful in notebooks. None = full data (use in gold.py).
    Returns (base_df, row_count). The count() fills the cache and doubles as
    the "Silver is not empty" check.
    """
    base = (
        spark.read.parquet(silver_path)
        .select(*GOLD_BASE_COLUMNS)
        .withColumn("request_date", F.to_date("created_at"))
    )

    if sample_pct is not None:
        base = base.filter(F.abs(F.hash("request_id")) % 100 < sample_pct)

    base = base.persist(StorageLevel.MEMORY_AND_DISK)
    row_count = base.count()
    if row_count == 0:
        raise ValueError(f"Silver is empty: {silver_path}")
    print(f"[Level-1 cache] Gold base cached: {row_count} rows", flush=True)
    return base, row_count


def write_table(df, path, partition_by=None):
    """Write one Gold table as Parquet (overwrite)."""
    writer = df.write.mode("overwrite")
    if partition_by:
        writer = writer.partitionBy(*partition_by)
    writer.parquet(path)
    print(f"  wrote {path}", flush=True)

In [6]:
# when we run both notebooks at the same time, use cores_max="6"

spark = create_spark_session(app_name="DAT535-Gold-MemberA", cores_max="6")

SAMPLE_PCT = 1      # 1% while developing; None = full data

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/13 15:43:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/13 15:43:55 WARN SparkConf: Total executor cores: 6 is not divisible by cores per executor: 4, the left cores: 2 will not be allocated


## 2. Load Silver – Level-1 cache

In [7]:
# LEVEL-1 CACHE (same as Lab 2: silver_df.cache())
base, n = load_gold_base(spark, DEFAULT_SILVER_PATH, sample_pct=SAMPLE_PCT)
base.printSchema()

[Stage 1:=======================================================> (33 + 1) / 34]

[Level-1 cache] Gold base cached: 227270 rows
root
 |-- request_id: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- closed_at: timestamp (nullable = true)
 |-- request_year: integer (nullable = true)
 |-- request_month: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- resolution_hours: double (nullable = true)
 |-- lifecycle_quality_status: string (nullable = true)
 |-- complaint_type: string (nullable = true)
 |-- agency: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- community_board: string (nullable = true)
 |-- incident_zip: string (nullable = true)
 |-- bbl: string (nullable = true)
 |-- pluto_match_status: string (nullable = true)
 |-- pluto_land_use: string (nullable = true)
 |-- pluto_building_class: string (nullable = true)
 |-- pluto_residential_units: long (nullable = true)
 |-- pluto_yearbuilt: long (nullable = true)
 |-- request_date: date (nullable = true)



## 3. Rules

In [ ]:
MIN_CLOSED_FOR_STATS = 30

## 4. Q1 – Where demand is concentrated

In [ ]:
def build_demand_by_area_month(base):
    """Requests per borough + community board per month."""
    return (
        base.groupBy("borough", "community_board", "request_year", "request_month")
        .agg(
            F.count("*").alias("total_requests"),
            F.countDistinct("complaint_type").alias("unique_complaint_types"),
            F.countDistinct("bbl").alias("unique_properties"),
        )
    )


def build_demand_by_complaint_borough(base):
    """Requests per complaint type in each borough, with rank and share."""
    counts = (
        base.filter(F.col("complaint_type").isNotNull())
        .groupBy("borough", "complaint_type")
        .agg(F.count("*").alias("total_requests"))
    )
    # Lab 3 window functions: rank and share inside each borough
    w_rank = Window.partitionBy("borough").orderBy(F.desc("total_requests"))
    w_total = Window.partitionBy("borough")
    return (
        counts
        .withColumn("rank_in_borough", F.row_number().over(w_rank))
        .withColumn("share_of_borough_pct",
                    F.round(F.col("total_requests") / F.sum("total_requests").over(w_total) * 100, 2))
    )

In [ ]:
demand_area = build_demand_by_area_month(base)
demand_area.groupBy("borough", "community_board").agg(F.sum("total_requests").alias("requests")) \
    .orderBy(F.desc("requests")).show(10)

demand_type = build_demand_by_complaint_borough(base)
demand_type.filter("rank_in_borough <= 3").orderBy("borough", "rank_in_borough").show(20, truncate=False)

## 5. Q2 – Slow or inconsistent resolution
Slow = high median / p90 hours. Inconsistent = high `cv_hours` (stddev ÷ mean).

In [ ]:
def resolution_stats():
    """Shared aggregation list: slow = high median/p90, inconsistent = high spread."""
    return [
        F.count("*").alias("total_requests"),
        F.count("resolution_hours").alias("closed_with_valid_time"),
        F.sum(F.when(F.col("status") != "Closed", 1).otherwise(0)).alias("num_unresolved"),
        F.round(F.percentile_approx("resolution_hours", 0.5), 2).alias("median_hours"),
        F.round(F.percentile_approx("resolution_hours", 0.9), 2).alias("p90_hours"),
        F.round(F.avg("resolution_hours"), 2).alias("avg_hours"),
        F.round(F.stddev("resolution_hours"), 2).alias("stddev_hours"),
    ]


def add_resolution_ratios(df):
    return (
        df
        .withColumn("pct_unresolved",
                    F.round(F.col("num_unresolved") / F.col("total_requests") * 100, 2))
        .withColumn(   # coefficient of variation = inconsistency (safe division)
            "cv_hours",
            F.when(F.col("avg_hours") > 0, F.round(F.col("stddev_hours") / F.col("avg_hours"), 2)))
        .withColumn("enough_data", F.col("closed_with_valid_time") >= MIN_CLOSED_FOR_STATS)
    )


def build_resolution_by_agency_complaint(base):
    return add_resolution_ratios(
        base.groupBy("agency", "complaint_type").agg(*resolution_stats())
    )


def build_resolution_by_borough_month(base):
    return add_resolution_ratios(
        base.groupBy("borough", "request_year", "request_month").agg(*resolution_stats())
    )

In [ ]:
res_agency = build_resolution_by_agency_complaint(base)
res_agency.filter("enough_data").orderBy(F.desc("median_hours")).show(10, truncate=False)
res_agency.filter("enough_data").orderBy(F.desc("cv_hours")).show(10, truncate=False)

## 6. Save to DEFAULT_GOLD_OUTPUT

In [ ]:
write_table(demand_area, f"{DEFAULT_GOLD_OUTPUT}/demand_by_area")
write_table(demand_type, f"{DEFAULT_GOLD_OUTPUT}/demand_by_type")
write_table(res_agency, f"{DEFAULT_GOLD_OUTPUT}/agency_resolution_time")

## 6. Clean up

In [ ]:
base.unpersist()
spark.stop()